<a href="https://colab.research.google.com/github/Bonesha-5/Group4_Formative3_Assignment/blob/main/bayesian.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries and Loading the Data

In [1]:
import csv

In [2]:
with open("IMDB Dataset.csv", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    reviews = list(reader)

print(f"Total reviews: {len(reviews)}")

Total reviews: 50000


## Selection Of Keywords

In [3]:
positive_keywords = [
    "excellent",
    "superb",
    "masterpiece",
    "outstanding"
]

negative_keywords = [
    "terrible",
    "awful",
    "horrible",
    "disappointing"
]

keywords = positive_keywords + negative_keywords

print(keywords)

['excellent', 'superb', 'masterpiece', 'outstanding', 'terrible', 'awful', 'horrible', 'disappointing']


**Calculating prior probabilities**

In [4]:
# PRIOR AND LIKELIHOOD CALCULATIONS

# Keep only reviews with valid sentiment labels
valid_reviews = []

for review in reviews:
    sentiment_val = review.get("sentiment")
    sentiment = (sentiment_val or "").strip().lower()

    if sentiment in ["positive", "negative"]:
        valid_reviews.append(review)

reviews = valid_reviews

# Count the total number of valid reviews
total_reviews = len(reviews)

# Count positive and negative reviews
positive_reviews = 0
negative_reviews = 0

for review in reviews:
    sentiment_val = review.get("sentiment")
    sentiment = (sentiment_val or "").strip().lower()

    if sentiment == "positive":
        positive_reviews += 1
    elif sentiment == "negative":
        negative_reviews += 1

# Calculate prior probabilities
prior_positive = positive_reviews / total_reviews if total_reviews > 0 else 0
prior_negative = negative_reviews / total_reviews if total_reviews > 0 else 0

# Display results
print("Total reviews:", total_reviews)
print("Positive reviews:", positive_reviews)
print("Negative reviews:", negative_reviews)

print("Prior P(Positive):", round(prior_positive, 4))
print("Prior P(Negative):", round(prior_negative, 4))

Total reviews: 50000
Positive reviews: 25000
Negative reviews: 25000
Prior P(Positive): 0.5
Prior P(Negative): 0.5


**Calculating keyword likelihood**

In [5]:
# Calculate likelihoods for each selected keyword

import re

# Store likelihood results
likelihood_results = []

for keyword in keywords:

    positive_count = 0
    negative_count = 0

    for review in reviews:
        sentiment = review["sentiment"].strip().lower()
        text = review["review"].lower()

        # Check whether the whole keyword appears in the review
        if re.search(r"\b" + re.escape(keyword) + r"\b", text):

            if sentiment == "positive":
                positive_count += 1

            elif sentiment == "negative":
                negative_count += 1

    # Calculate likelihoods for both classes
    likelihood_positive = positive_count / positive_reviews
    likelihood_negative = negative_count / negative_reviews

    # Store the calculated values
    likelihood_results.append([
        keyword,
        positive_count,
        negative_count,
        likelihood_positive,
        likelihood_negative
    ])

**Prior probabilities and likelihood results**

In [6]:
# Display the calculated prior probabilities

print("PRIOR PROBABILITIES")
print("-" * 40)

print(f"P(Positive): {prior_positive:.4f}")
print(f"P(Negative): {prior_negative:.4f}")

print("\nKEYWORD LIKELIHOODS")
print("-" * 85)

# Display likelihoods in a table
print(
    f"{'Keyword':<18}"
    f"{'Positive Count':<18}"
    f"{'Negative Count':<18}"
    f"{'P(W|Positive)':<16}"
    f"{'P(W|Negative)':<16}"
)

print("-" * 85)

for row in likelihood_results:
    keyword, pos_count, neg_count, likelihood_pos, likelihood_neg = row

    print(
        f"{keyword:<18}"
        f"{pos_count:<18}"
        f"{neg_count:<18}"
        f"{likelihood_pos:<16.4f}"
        f"{likelihood_neg:<16.4f}"
    )

PRIOR PROBABILITIES
----------------------------------------
P(Positive): 0.5000
P(Negative): 0.5000

KEYWORD LIKELIHOODS
-------------------------------------------------------------------------------------
Keyword           Positive Count    Negative Count    P(W|Positive)   P(W|Negative)   
-------------------------------------------------------------------------------------
excellent         2868              684               0.1147          0.0274          
superb            1026              173               0.0410          0.0069          
masterpiece       878               329               0.0351          0.0132          
outstanding       618               125               0.0247          0.0050          
terrible          383               2318              0.0153          0.0927          
awful             284               2599              0.0114          0.1040          
horrible          326               1775              0.0130          0.0710          
disappoint

Marginal probability (how often each keyword appears in all reviews)

In [7]:
# Calculate the marginal probability P(word) for each keyword
marginal_results = []

print(f"{'Keyword':<16}{'Reviews':>10}{'P(word)':>10}")
print("-" * 36)

for row in likelihood_results:
    keyword, pos_count, neg_count, likelihood_pos, likelihood_neg = row

    # Reviews that contain the keyword (positive + negative)
    word_count = pos_count + neg_count
    marginal = word_count / total_reviews

    marginal_results.append([keyword, word_count, marginal])

    print(f"{keyword:<16}{word_count:>10}{marginal:>10.4f}")

Keyword            Reviews   P(word)
------------------------------------
excellent             3552    0.0710
superb                1199    0.0240
masterpiece           1207    0.0241
outstanding            743    0.0149
terrible              2701    0.0540
awful                 2883    0.0577
horrible              2101    0.0420
disappointing          786    0.0157


Posterior probability (how likely a review is positive when it contains the keyword.)

In [8]:
# Bayes' theorem: P(Positive | word) = P(word | Positive) * P(Positive) / P(word)
posterior_results = []

for i in range(len(likelihood_results)):
    keyword = likelihood_results[i][0]
    likelihood_pos = likelihood_results[i][3]
    marginal = marginal_results[i][2]

    posterior = (likelihood_pos * prior_positive) / marginal

    posterior_results.append([keyword, posterior])

    print(f"P(Positive | {keyword}) = {posterior:.4f}")

P(Positive | excellent) = 0.8074
P(Positive | superb) = 0.8557
P(Positive | masterpiece) = 0.7274
P(Positive | outstanding) = 0.8318
P(Positive | terrible) = 0.1418
P(Positive | awful) = 0.0985
P(Positive | horrible) = 0.1552
P(Positive | disappointing) = 0.2176


Check (compare the Bayes result with a direct count.)

In [9]:
# Check: Bayes result should equal positive reviews with the word / all reviews with the word
print(f"{'Keyword':<16}{'Bayes':>8}{'Direct':>8}{'Match':>8}")
print("-" * 40)

for i in range(len(likelihood_results)):
    keyword, pos_count, neg_count = likelihood_results[i][0:3]

    direct = pos_count / (pos_count + neg_count)
    bayes = posterior_results[i][1]
    match = "Yes" if abs(bayes - direct) < 1e-9 else "No"

    print(f"{keyword:<16}{bayes:>8.4f}{direct:>8.4f}{match:>8}")

Keyword            Bayes  Direct   Match
----------------------------------------
excellent         0.8074  0.8074     Yes
superb            0.8557  0.8557     Yes
masterpiece       0.7274  0.7274     Yes
outstanding       0.8318  0.8318     Yes
terrible          0.1418  0.1418     Yes
awful             0.0985  0.0985     Yes
horrible          0.1552  0.1552     Yes
disappointing     0.2176  0.2176     Yes


In [10]:
print("BAYES' THEOREM RESULTS")
print("-" * 61)
print(
    f"{'Keyword':<16}"
    f"{'Prior':<10}"
    f"{'Likelihood':<14}"
    f"{'Marginal':<12}"
    f"{'Posterior':<12}"
)
print("-" * 61)

for i in range(len(likelihood_results)):
    keyword = likelihood_results[i][0]
    likelihood_pos = likelihood_results[i][3]
    marginal = marginal_results[i][2]
    posterior = posterior_results[i][1]

    print(
        f"{keyword:<16}"
        f"{prior_positive:<10.4f}"
        f"{likelihood_pos:<14.4f}"
        f"{marginal:<12.4f}"
        f"{posterior:<12.4f}"
    )

BAYES' THEOREM RESULTS
-------------------------------------------------------------
Keyword         Prior     Likelihood    Marginal    Posterior   
-------------------------------------------------------------
excellent       0.5000    0.1147        0.0710      0.8074      
superb          0.5000    0.0410        0.0240      0.8557      
masterpiece     0.5000    0.0351        0.0241      0.7274      
outstanding     0.5000    0.0247        0.0149      0.8318      
terrible        0.5000    0.0153        0.0540      0.1418      
awful           0.5000    0.0114        0.0577      0.0985      
horrible        0.5000    0.0130        0.0420      0.1552      
disappointing   0.5000    0.0068        0.0157      0.2176      
